In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GridSearchCV, cross_val_predict
)
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_squared_error, accuracy_score,
    precision_score, recall_score, f1_score
)

from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

In [2]:
def compute_metrics(y_true, y_pred, label=""):
    mse  = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average="weighted", zero_division=0)
    rec  = recall_score(y_true, y_pred, average="weighted", zero_division=0)
    f1   = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "Set": label,
        "MSE": round(mse, 4),
        "RMSE": round(rmse, 4),
        "Accuracy": round(acc, 4),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1": round(f1, 4)
    }

In [3]:
def normalize_target(price):
    if price > 7300:
        return 1.0
    elif price < 2160:
        return 0.0
    elif 2160 <= price <= 7300:
        return (price - 2160) / (7300 - 2160)
    else:
        return 0.0 

In [4]:
data = pd.read_csv('ostrava_housing.csv')
data['target_Normalized']=data['target'].apply(normalize_target)

# decreasing
columns_to_keep = [
    'ownership', 'energy_label', 'state', 'equipment', 'crime_index', 'quality_index'
]
for col in columns_to_keep:
    min_val = data[col].min()
    max_val = data[col].max()
    data[f"{col}_Normalized"] = 1- ((data[col] - min_val) / (max_val - min_val))

# increasing
columns_to_keep = [
    'area', 'rooms'
]
for col in columns_to_keep:
    min_val = data[col].min()
    max_val = data[col].max()
    data[f"{col}_Normalized"] = (data[col] - min_val) / (max_val - min_val)

preprocessed_data = data[[col for col in data.columns if col.endswith('_Normalized')]]
df_final = preprocessed_data.iloc[:,[2,3,4,5,6,1,8,7,0]]
df_final = df_final.rename(columns={'target_Normalized': 'target'})
df_final['target'] = df_final.apply(lambda row: 0 if (row['target']<0.5)else 1, axis=1)

X_train, X_test = train_test_split(df_final, test_size=0.2, random_state=42)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=df_final.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=df_final.columns)

X_train = X_train_scaled.iloc[:, :-1]
X_test = X_test_scaled.iloc[:, :-1]
y_train = X_train_scaled.iloc[:, -1]
y_test = X_test_scaled.iloc[:, -1]

In [5]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

models = {
    "Naive Bayes": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", GaussianNB())
        ]),
        "params": {
            "clf__var_smoothing": np.logspace(-12, -6, 7)
        }
    },

    "Logistic Regression": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=42))
        ]),
        "params": {
            "clf__C": [0.001, 0.01, 0.1, 1, 10, 100],
            "clf__solver": ["lbfgs", "liblinear"],
            "clf__penalty": ["l2"]
        }
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", RandomForestClassifier(random_state=42))
        ]),
        "params": {
            "clf__n_estimators": [50, 100, 200],
            "clf__max_depth": [None, 5, 10],
            "clf__min_samples_split": [2, 5],
            "clf__min_samples_leaf": [1, 2]
        }
    },

    "Neural Network": {
        "pipeline": Pipeline([
            ("scaler", MinMaxScaler()),
            ("clf", MLPClassifier(max_iter=500, random_state=42))
        ]),
        "params": {
            "clf__hidden_layer_sizes": [(64,), (128,), (64, 32), (128, 64)],
            "clf__activation": ["relu", "tanh"],
            "clf__alpha": [0.0001, 0.001, 0.01],
            "clf__learning_rate_init": [0.001, 0.01]
        }
    }
}

In [6]:
all_results = []

for name, config in models.items():
    print(f"\n{'='*60}")
    print(f"  Model: {name}")
    print(f"{'='*60}")

    gs = GridSearchCV(
        estimator=config["pipeline"],
        param_grid=config["params"],
        cv=cv,
        scoring="f1_weighted",
        n_jobs=-1,
        verbose=0
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_
    print(f"  The best params : {gs.best_params_}")
    print(f"  CV F1 (val. set)   : {gs.best_score_:.4f}")

    y_train_pred = cross_val_predict(best_model, X_train, y_train, cv=cv)
    train_metrics = compute_metrics(y_train, y_train_pred, label="Train (CV)")

    y_test_pred = best_model.predict(X_test)
    test_metrics = compute_metrics(y_test, y_test_pred, label="Test")

    for m in [train_metrics, test_metrics]:
        print(f"\n  [{m['Set']}]")
        for k, v in m.items():
            if k != "Set":
                print(f"    {k:<12}: {v}")

    all_results.append({"Model": name, **train_metrics})
    all_results.append({"Model": name, **test_metrics})


  Model: Naive Bayes
  The best params : {'clf__var_smoothing': 1e-12}
  CV F1 (val. set)   : 0.8311

  [Train (CV)]
    MSE         : 0.1721
    RMSE        : 0.4148
    Accuracy    : 0.8279
    Precision   : 0.8378
    Recall      : 0.8279
    F1          : 0.8314

  [Test]
    MSE         : 0.1593
    RMSE        : 0.3991
    Accuracy    : 0.8407
    Precision   : 0.842
    Recall      : 0.8407
    F1          : 0.8413

  Model: Logistic Regression
  The best params : {'clf__C': 100, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
  CV F1 (val. set)   : 0.8504

  [Train (CV)]
    MSE         : 0.1448
    RMSE        : 0.3805
    Accuracy    : 0.8552
    Precision   : 0.8509
    Recall      : 0.8552
    F1          : 0.8508

  [Test]
    MSE         : 0.1534
    RMSE        : 0.3917
    Accuracy    : 0.8466
    Precision   : 0.84
    Recall      : 0.8466
    F1          : 0.8401

  Model: Random Forest
  The best params : {'clf__max_depth': None, 'clf__min_samples_leaf': 1, 'clf__min_

In [7]:
print(f"\n\n{'='*60}")
print("Metrics")
print(f"{'='*60}")
df_results = pd.DataFrame(all_results)
df_results = df_results.set_index(["Model", "Set"])
print(df_results.to_string())



Metrics
                                   MSE    RMSE  Accuracy  Precision  Recall      F1
Model               Set                                                            
Naive Bayes         Train (CV)  0.1721  0.4148    0.8279     0.8378  0.8279  0.8314
                    Test        0.1593  0.3991    0.8407     0.8420  0.8407  0.8413
Logistic Regression Train (CV)  0.1448  0.3805    0.8552     0.8509  0.8552  0.8508
                    Test        0.1534  0.3917    0.8466     0.8400  0.8466  0.8401
Random Forest       Train (CV)  0.1100  0.3317    0.8900     0.8879  0.8900  0.8869
                    Test        0.1032  0.3213    0.8968     0.8943  0.8968  0.8938
Neural Network      Train (CV)  0.1396  0.3736    0.8604     0.8573  0.8604  0.8582
                    Test        0.1327  0.3643    0.8673     0.8628  0.8673  0.8613
